In [1]:
import pandas as pd

In [2]:
DATA_PATH = "wfp_food_prices_database.csv"   # <-- change to your file name
SEP = ","                           # use "\t" if your file is tab-separated
PRICE_TYPE = "Retail"               # keep one price type; set None to keep all
MIN_MONTHS = 60                     # a series needs at least this many months
MIN_COMPLETENESS = 0.80             # and at least this share of months filled

In [3]:
df = pd.read_csv(DATA_PATH, sep=SEP, low_memory=False)

keep = ["adm0_name", "adm1_name", "mkt_name", "cm_name", "pt_name",
        "um_name", "cur_name", "mp_month", "mp_year", "mp_price"]
df = df[keep].copy()

df["date"] = pd.to_datetime(
    dict(year=df["mp_year"], month=df["mp_month"], day=1), errors="coerce"
)
df["mp_price"] = pd.to_numeric(df["mp_price"], errors="coerce")
df = df.dropna(subset=["date", "mp_price"])
df = df[df["mp_price"] > 0]

if PRICE_TYPE:
    df = df[df["pt_name"] == PRICE_TYPE]

print(f"Rows after cleaning: {len(df):,}")
print(f"Countries: {df['adm0_name'].nunique()}")

Rows after cleaning: 1,417,458
Countries: 89


In [4]:
series_keys = ["adm0_name", "mkt_name", "cm_name", "um_name"]

g = df.groupby(series_keys)["date"]
series = g.agg(first="min", last="max", n_obs="nunique").reset_index()
series["span_months"] = (
    (series["last"].dt.year - series["first"].dt.year) * 12
    + (series["last"].dt.month - series["first"].dt.month) + 1
)
series["completeness"] = series["n_obs"] / series["span_months"]

series["viable"] = (
    (series["n_obs"] >= MIN_MONTHS) & (series["completeness"] >= MIN_COMPLETENESS)
)

In [5]:
country = (
    series.groupby("adm0_name")
    .agg(
        n_series=("viable", "size"),
        viable_series=("viable", "sum"),
        median_months=("n_obs", "median"),
        median_completeness=("completeness", "median"),
        first_date=("first", "min"),
        last_date=("last", "max"),
    )
    .sort_values("viable_series", ascending=False)
)
country["pct_viable"] = (country["viable_series"] / country["n_series"]).round(2)

print("\nTop 15 countries by number of viable series")
print(country.head(15).to_string())

country.to_csv("country_coverage.csv")


Top 15 countries by number of viable series
                                  n_series  viable_series  median_months  median_completeness first_date  last_date  pct_viable
adm0_name                                                                                                                      
Bassas da India                       1260            742           81.0             0.909091 1994-01-01 2020-06-01        0.59
Rwanda                                3952            410           35.0             0.791667 2008-01-01 2020-08-01        0.10
Zambia                                 529            331           92.0             0.930693 2003-01-01 2020-06-01        0.63
Syrian Arab Republic                  3583            280           13.0             1.000000 2011-04-01 2020-08-01        0.08
Mali                                  1901            279            4.0             1.000000 2003-01-01 2020-07-01        0.15
Lao People's Democratic Republic       408            266  

In [8]:
CHOSEN = country.index[0]           # <-- replace with e.g. "India" once you decide
print(f"\nDrill-down: {CHOSEN}")

sub = series[(series["adm0_name"] == CHOSEN) & series["viable"]]

by_commodity = (
    sub.groupby("cm_name")
    .agg(
        n_markets=("mkt_name", "nunique"),
        median_months=("n_obs", "median"),
        median_completeness=("completeness", "median"),
        first=("first", "min"),
        last=("last", "max"),
    )
    .sort_values("n_markets", ascending=False)
)
print(by_commodity.head(15).to_string())
by_commodity.to_csv(f"{CHOSEN.lower().replace(' ', '_')}_commodity_coverage.csv")

# Units: make sure each commodity uses one unit, or convert later
units = sub.groupby("cm_name")["um_name"].nunique()
mixed = units[units > 1]
if len(mixed):
    print("\nCommodities reported in more than one unit (needs harmonizing):")
    print(mixed.to_string())


Drill-down: Bassas da India
                             n_markets  median_months  median_completeness      first       last
cm_name                                                                                         
Rice - Retail                       57          109.0             0.943396 1994-01-01 2020-06-01
Potatoes - Retail                   56           92.0             0.929293 2000-01-01 2020-06-01
Lentils - Retail                    54           81.0             0.931034 2013-04-01 2020-06-01
Onions - Retail                     54           91.0             0.919192 2000-01-01 2020-06-01
Salt (iodised) - Retail             50           90.0             0.909091 2012-04-01 2020-06-01
Tomatoes - Retail                   49           90.0             0.909091 2012-04-01 2020-06-01
Wheat flour - Retail                49           90.0             0.909091 2010-01-01 2020-06-01
Milk (pasteurized) - Retail         48           92.0             0.929293 2012-04-01 2020-06-01
T